# <mark style="display:block; background:#d1c4e9; color:#1a1a1a; padding:6px 12px; border-radius:4px">8주차 · FastAPI 로 서빙 — 투자자 번호를 보내면 추천을 돌려주는 API</mark>

7주차에 300명의 추천을 **파일 하나**(`recommendations.csv`)로 만들었습니다. 그런데 아직 아무도 이 파일을 쓰지 않습니다.

실제 서비스에서는 모바일 앱 · 상담 직원 화면 · 챗봇이 모두 추천을 필요로 합니다. 이들이 각자 CSV 를 열게 하지 않고, **「U0001 추천 줘」라고 요청하면 답을 돌려주는 창구** 하나를 둡니다. 이것이 **API** 입니다.

| 주차 | 만든 것 | 누가 쓰나 |
|---|---|---|
| 7주차 | `output/recommendations.csv` — 300명 × 10개 | 아직 아무도 |
| **8주차 (오늘)** | **추천 API** — 투자자 번호를 보내면 추천을 돌려준다 | 앱 · 상담 화면 · 챗봇 같은 다른 프로그램 |
| 9주차 | Streamlit 화면 — 이 추천을 사람이 보는 화면으로 | 사람 |

API 는 **`recommend_api.py` 파일 하나**에 들어 있습니다. 이 노트북은 그 API 를 **켜고 → 요청을 보내고 → 답을 확인**합니다.

**미리 말하면** — 모델도 점수(0.3899)도 그대로입니다. API 는 학습하지 않고, 7주차 파일을 **꺼내 주기만** 합니다(서빙).

---

### 오늘의 구성 — 50분 (PPT + 노트북)

| 시간 | 어디서 | 하는 일 |
|---|---|---|
| 개념 30분 | PPT | API · 요청과 응답 · 엔드포인트 · JSON · 상태 코드 · FastAPI · Swagger · 배포 개념 |
| 실습 20분 | **이 노트북** | 모두 실행 → 8.2 서버 켜기 → 8.3 U0001 요청 → 8.4 Swagger (강사 시연) → 8.5 없는 번호 → 🗣 함께 보기 |

### 이 노트북 읽는 법 — 🎯 와 📖

- **🎯 수업** — 수업 시간에 셀을 실행하고 **결과를 함께 읽는** 절입니다
  - 수업에서는 설명 글을 다 읽지 않습니다. 셀 실행과 「결과 읽는 법」만 봅니다
- **📖 혼자 읽기** — 수업에서는 건너뛰고 **복습할 때 읽는** 부분입니다
  - 절 전체가 📖 인 곳(8.1 · 8.6 · 8.7 · 8.8)과, 🎯 절 안의 「왜 배우나 · 배경 지식 · 어떻게 동작하나」 블록이 여기에 속합니다

⏱ **실습을 시작하면 먼저 메뉴의 `런타임 → 모두 실행` 을 눌러 두세요.** 데이터 받기 · 7주차 결과 파일 만들기 · 서버 켜기까지 20초쯤 걸립니다.

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">시작하기 · 이 노트북을 여는 법</mark>

1. https://colab.research.google.com/github/welovecherry/recsys/blob/main/notebooks/08_api_serving.ipynb
2. 구글 계정으로 로그인합니다.
3. **경고창이 뜨면 `Run anyway` 를 누릅니다.**
4. **아래 셀의 ▶ 버튼을 누릅니다.** 실습 자료를 받아옵니다. 10초쯤 걸립니다.
5. 그다음부터는 위에서 아래로 셀을 하나씩 실행하면 됩니다.

> ⚠ **고친 내용을 남기려면** 메뉴에서 `파일 → 드라이브에 사본 저장` 을 눌러 주세요.

---

**이 노트북이 쓰는 것**

| 무엇 | 쓰임 |
|---|---|
| `recommend_batch.py` | 7주차 스크립트 — 결과 파일 `output/recommendations.csv` 를 만든다 |
| `recommend_api.py` | **오늘의 산출물** — 결과 파일을 읽어 추천을 돌려주는 API |

⚠ **지난주 노트북을 먼저 돌릴 필요가 없습니다.** 7주차 결과 파일도 이 노트북이 다시 만듭니다.

⚠ **따로 설치할 것이 없습니다.** 오늘 쓰는 도구는 Colab 에 처음부터 들어 있습니다.

---

## <mark style="display:block; background:#c8e6c9; color:#1a1a1a; padding:6px 12px; border-radius:4px">먼저 이 셀부터 실행하세요  ▶</mark>

In [1]:
# 이 셀에서 하는 일 — Colab 에서 열었으면 실습 자료를 내려받는다
# 왜 하나 — Colab 은 열 때마다 빈 컴퓨터라 데이터·recsys.py 가 없다. 내 컴퓨터면 그냥 넘어간다

import os          # 폴더를 만들고 옮겨 다니는 도구
import sys         # 지금 파이썬이 어떤 환경인지 알려 주는 도구
import subprocess  # 터미널 명령을 파이썬에서 대신 실행해 주는 도구

if "google.colab" in sys.modules:                    # Colab 이면 이 안이 실행된다
    if os.path.exists("/content/recsys"):            # 전에 받아 둔 것이 있으면 최신으로
        subprocess.run(["git", "-C", "/content/recsys", "pull", "-q", "--ff-only"])
    else:                                            # 처음이면 통째로 내려받는다
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/welovecherry/recsys.git", "/content/recsys"])
    os.chdir("/content/recsys/notebooks")            # 노트북 폴더 안으로 이동
    print("준비 끝 —", os.getcwd(), "· 아래 셀부터 차례로 실행하세요.")
else:
    print("내 컴퓨터에서 실행 중입니다 —", os.getcwd(), "· 따로 받아올 것이 없습니다.")

내 컴퓨터에서 실행 중입니다 — /Users/hong/workspaces/org_physical-spark/course-recsys/notebooks · 따로 받아올 것이 없습니다.


### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">오늘 나오는 말 일곱 개</mark>

**1\~5번은 API 의 기본, 6\~7번은 오늘 쓰는 도구입니다.** 괄호 안의 절에서 예와 함께 다시 설명합니다.

1. **API (Application Programming Interface)** — 다른 프로그램이 **주소로 요청하면 답을 돌려주는 창구** (8.2)
   - 사람이 아니라 **프로그램**이 쓰는 창구입니다
   - 예 — 은행 앱이 잔고를 보여 줄 때도 뒤에서 은행의 API 를 부릅니다

2. **요청 (request) · 응답 (response)** — 「U0001 추천 줘」가 요청, 「여기 10개」가 응답 (8.3)
   - 웹 브라우저로 주소를 여는 것도 요청을 보내고 응답을 받아 보여 주는 일입니다

3. **엔드포인트 (endpoint)** — 요청을 받는 **주소 하나** (8.1)
   - 오늘은 두 개 — `/health` (켜져 있나) · `/recommend/U0001` (U0001 의 추천)

4. **JSON (JavaScript Object Notation)** — API 가 답을 돌려줄 때 쓰는 **글자 형식** (8.3)
   - 파이썬 딕셔너리와 거의 같은 모양입니다 — `{"user_id": "U0001", "persona": "안정형"}`

5. **상태 코드 (status code)** — 응답 맨 앞에 붙는 **결과 번호** (8.5)
   - **200** = 성공 · **404** = 찾는 것이 없음 · **422** = 요청이 규칙에 맞지 않음

6. **FastAPI** — 파이썬으로 API 를 **몇 줄 만에** 만드는 도구 (8.1)
   - Colab 에 처음부터 들어 있습니다

7. **Swagger (스웨거)** — FastAPI 가 **자동으로 만들어 주는 API 설명서 겸 시험 화면** (8.4)
   - 주소 끝에 `/docs` 를 붙이면 열립니다. 코드 없이 버튼으로 요청을 보내 볼 수 있습니다

---

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 준비 — 7주차 결과 파일 만들기</mark>

**여기서 하는 일** — 7주차 스크립트를 한 번 실행해 결과 파일을 만듭니다. API 가 읽을 파일입니다.

- Colab 은 열 때마다 빈 컴퓨터라서, 지난주에 만든 파일이 남아 있지 않습니다.

In [2]:
# 이 셀에서 하는 일 — 7주차 스크립트로 결과 파일을 만든다
# 왜 하나 — 오늘 API 가 읽을 파일이 output/recommendations.csv 이다

import sys                                  # 파이썬이 파일을 찾는 경로를 다루는 도구
sys.path.insert(0, ".")
sys.path.insert(0, "notebooks")

!python recommend_batch.py

① 데이터 점검 — 여섯 가지 모두 0건 · 통과


② 학습 — 거래 기록 8,086건 · 투자자 300명


③ 일괄 추천 — 3,000줄
④ 규격 검사 — 문제 0개 · 통과


⑤ 저장 — output/recommendations.csv · 300명 × 10개


**결과 읽는 법**

- 7주차 7.3 과 같은 다섯 줄입니다. 마지막 줄 **⑤ 저장** 이 나오면 결과 파일이 준비된 것입니다.
  - 데이터 점검 통과 → 거래 기록 8,086건으로 학습 → 3,000줄 → 규격 검사 통과 → 저장

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.1 API 파일 읽기 — 엔드포인트 두 개</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 한 장으로 보여 줍니다. 복습할 때 아래 셀을 실행해 코드를 직접 읽어 보세요.

**8.1 에서 하는 일** — `recommend_api.py` 의 코드를 화면에 띄워 **엔드포인트 두 개**가 어디 있는지 봅니다.

**왜 배우나**
- 오늘 켜는 API 가 무엇을 받고 무엇을 돌려주는지 알아야 요청을 보낼 수 있습니다.
- FastAPI 코드는 짧습니다. 비개발자도 「어떤 주소에서 무엇을 주는지」는 읽을 수 있습니다.

**알아 둘 배경 지식**
- **`@app.get("/health")`** — 「`/health` 주소로 **가져와라(GET)** 요청이 오면 바로 아래 함수를 실행하라」는 표시
  - `@` 로 시작하는 줄을 **데코레이터(decorator)** 라고 합니다. 함수 위에 붙여 「이 함수를 어디에 쓸지」를 알려 줍니다
- **`/recommend/{user_id}`** — 중괄호 자리에 투자자 번호가 들어갑니다. `/recommend/U0001` 이면 `user_id` 가 `"U0001"`
- **`k: int = Query(10, ge=1, le=10)`** — 주소 뒤 `?k=5` 로 몇 개 받을지 정한다. 안 적으면 10, 1 \~ 10 만 허용
  - `ge` = greater or equal(이상), `le` = less or equal(이하)
- **`raise HTTPException(status_code=404, ...)`** — 「찾는 것이 없다」는 응답을 돌려주고 함수를 끝낸다
- **`return {...}`** — 딕셔너리를 돌려주면 FastAPI 가 **JSON 으로 바꿔서** 응답합니다

**어떻게 동작하나**
1. API 가 켜질 때 결과 파일을 **한 번** 읽어 `추천표` 에 담는다
2. `/health` 요청 → `{"status": "ok", "rows": 3000}`
3. `/recommend/U0001` 요청 → `추천표` 에서 U0001 의 줄만 골라 순위대로 k개 → JSON
4. 결과 파일에 없는 번호면 → **404**

In [3]:
# 이 셀에서 하는 일 — API 파일의 코드를 화면에 띄운다
# 왜 하나 — 엔드포인트 두 개가 무엇을 받고 무엇을 돌려주는지 읽는다

with open("recommend_api.py", encoding="utf-8") as 파일:   # with = 다 읽으면 파일을 저절로 닫는다
    코드 = 파일.read()
print(코드[코드.index("app = FastAPI"):])                    # 앞쪽 설명은 건너뛰고 API 부분만

app = FastAPI(title="추천 API",
              description="7주차 결과 파일(recommendations.csv)을 읽어 투자자별 추천을 돌려줍니다.")


@app.get("/health")
def health():
    """API 가 켜져 있는지, 결과 파일을 몇 줄 읽었는지 알려 준다."""
    return {"status": "ok", "rows": len(추천표)}


@app.get("/recommend/{user_id}")
def recommend(user_id: str, k: int = Query(10, ge=1, le=10, description="몇 개를 받을지 (1 ~ 10)")):
    """투자자 번호를 받아, 위험도 상한을 지킨 추천을 순위대로 k개 돌려준다."""
    줄들 = 추천표[추천표["user_id"] == user_id].sort_values("rank").head(k)
    if 줄들.empty:                                              # 결과 파일에 없는 번호
        raise HTTPException(status_code=404, detail=f"{user_id} 는 추천이 없는 투자자입니다")
    추천 = []
    for _, 줄 in 줄들.iterrows():
        추천.append({"rank": int(줄["rank"]), "item_id": 줄["item_id"], "item_name": 줄["item_name"],
                     "risk_level": int(줄["risk_level"]), "score": float(줄["score"])})
    return {"user_id": user_id, "persona": 줄들["persona"].iloc[0], "recommendations": 추천}



**결과 읽는 법**

- `@app.get("/health")` 아래가 첫째 엔드포인트, `@app.get("/recommend/{user_id}")` 아래가 둘째 엔드포인트입니다.
- `recommend` 함수 안에 **학습하는 줄이 없습니다.** 파일에서 고르고(`추천표[...]`), 순위대로 자르고(`head(k)`), 돌려줄(`return`) 뿐입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.2 서버 켜기 — API 를 켜 두고 살아 있는지 확인</mark>

**8.2 에서 하는 일** — API 를 **서버**로 켜 두고, `/health` 에 요청을 보내 「살아 있다」는 답을 받습니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- API 는 파일만 있다고 동작하지 않습니다. 누군가 **켜 두어야** 요청을 받을 수 있습니다. 켜 둔 프로그램이 **서버(server)** 입니다.

**왜 중요한가**
- 실제 서비스에서는 「API 가 죽었는지」를 몇 초마다 확인합니다. `/health` 같은 주소가 그 확인용입니다.
  - 죽었으면 자동으로 다시 켜거나 담당자에게 알림을 보냅니다.
- 고객 앱은 API 가 꺼져 있으면 추천을 하나도 못 보여 줍니다.

**알아 둘 배경 지식**
- **서버 (server)** — 계속 켜져서 요청을 기다리는 프로그램. 셀이 끝나도 뒤에서 돌아갑니다
- **`localhost`** — 「바로 이 컴퓨터」를 뜻하는 주소. 오늘 API 는 이 Colab 컴퓨터 안에서만 부를 수 있습니다
- **포트 (port)** — 한 컴퓨터 안에서 서버마다 붙이는 번호. 오늘은 8765
- **주소 읽는 법** — `http://localhost:8765/health`
  - `http://` 요청 방식 · `localhost` 어느 컴퓨터 · `8765` 어느 서버 · `/health` 어느 엔드포인트
- **uvicorn (유비콘)** — FastAPI 코드를 실제 서버로 켜 주는 도구. Colab 에 들어 있습니다
- **requests** — 파이썬에서 요청을 보내는 도구. 브라우저가 하는 일을 코드로 합니다

**어떻게 동작하나**
1. 먼저 `/health` 에 요청을 보내 본다 — 이미 켜져 있으면 다시 켜지 않는다
2. 꺼져 있으면 uvicorn 으로 `recommend_api.py` 를 **뒤에서(백그라운드)** 켠다
3. 켜질 때까지 0.5초마다 다시 물어본다
4. 켜지면 `/health` 의 응답을 보여 준다

---

#### 자세히

- `subprocess.Popen` — 다른 프로그램을 **켜 두고 기다리지 않는** 도구입니다. 7주차의 `!python` 은 스크립트가 끝날 때까지 기다렸지만, 서버는 끝나지 않는 프로그램이라 기다리면 안 됩니다.
- Colab 연결이 끊기면 서버도 함께 꺼집니다. 그때는 `런타임 → 모두 실행` 을 다시 누르면 됩니다.

In [4]:
# 이 셀에서 하는 일 — API 서버를 뒤에서 켜고, /health 로 살아 있는지 확인한다
# 왜 하나 — 서버가 켜져 있어야 요청을 보낼 수 있다

import subprocess, time                      # 다른 프로그램을 켜는 도구 · 기다리는 도구
import requests                              # 요청을 보내는 도구
import pandas as pd                          # 표를 다루는 도구

포트 = 8765                                   # 오늘 API 서버의 번호
주소 = f"http://localhost:{포트}"             # 이 컴퓨터 안의 8765 번 서버

def 켜져_있나():
    """/health 에 요청을 보내 200 이 오면 True."""
    try:
        return requests.get(주소 + "/health", timeout=1).status_code == 200
    except requests.exceptions.RequestException:   # 아직 안 켜졌으면 연결 오류가 난다
        return False

if not 켜져_있나():                                      # 꺼져 있을 때만 켠다
    subprocess.Popen([sys.executable, "-m", "uvicorn", "recommend_api:app", "--port", str(포트)],
                     stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)   # 서버가 찍는 글자는 버린다
    for _ in range(40):                                  # 최대 20초 기다린다
        if 켜져_있나():
            break
        time.sleep(0.5)

응답 = requests.get(주소 + "/health")
print("상태 코드 :", 응답.status_code)
print("응답 :", 응답.json())

상태 코드 : 200
응답 : {'status': 'ok', 'rows': 3000}


**결과 읽는 법**

- **상태 코드 200** — 요청이 성공했습니다. API 가 켜져 있습니다.
- **응답 `{'status': 'ok', 'rows': 3000}`**
  - `status: ok` — 살아 있다
  - `rows: 3000` — 켜질 때 결과 파일 3,000줄(300명 × 10개)을 읽었다
- 이제 이 서버는 셀이 끝나도 뒤에서 계속 요청을 기다립니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.3 요청 보내기 — U0001 의 추천을 받는다</mark>

**8.3 에서 하는 일** — `/recommend/U0001` 에 요청을 보내 **JSON** 으로 답을 받고, 표로 바꿔 봅니다. 7주차 파일과 같은지도 확인합니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- 오늘의 핵심 장면입니다. 다른 프로그램은 파일을 몰라도 **주소 하나**로 추천을 받습니다.

**왜 중요한가**
- 앱 · 상담 화면 · 챗봇이 모두 **같은 주소**를 부르면, 모두 **같은 추천**을 보여 줍니다.
  - 각자 파일을 읽으면 누군가는 어제 파일을, 누군가는 오늘 파일을 읽을 수 있습니다.
- 추천 방식이 바뀌어도(9주차 이후 모델을 바꾸더라도) **주소와 답의 모양만 그대로면** 앱은 고칠 필요가 없습니다.

**알아 둘 배경 지식**
- **JSON 과 딕셔너리**

| JSON (API 가 보낸 글자) | 파이썬에서 받으면 |
|---|---|
| `{"user_id": "U0001"}` | 딕셔너리 `{"user_id": "U0001"}` |
| `[{"rank": 1}, {"rank": 2}]` | 딕셔너리들의 목록 |

- `응답.json()` — 받은 JSON 글자를 파이썬 딕셔너리로 바꿉니다
- `pd.DataFrame(목록)` — 딕셔너리들의 목록을 표로 바꿉니다

**어떻게 동작하나**
1. `requests.get(주소 + "/recommend/U0001")` — 요청을 보낸다
2. API 가 결과 파일에서 U0001 의 10줄을 골라 JSON 으로 돌려준다
3. `응답.json()` 으로 딕셔너리로 바꾼다
4. 그 안의 `recommendations` 목록을 표로 바꿔 본다

In [5]:
# 이 셀에서 하는 일 — U0001 의 추천을 요청하고, 받은 JSON 을 표로 본다
# 왜 하나 — 다른 프로그램이 주소 하나로 추천을 받는 장면을 직접 본다

응답 = requests.get(주소 + "/recommend/U0001")            # U0001 의 추천을 달라는 요청
답 = 응답.json()                                          # JSON 글자 → 파이썬 딕셔너리

print("상태 코드 :", 응답.status_code)
print("답에 든 것 :", list(답.keys()))
print(f"투자자 {답['user_id']} · 투자성향 {답['persona']} · 추천 {len(답['recommendations'])}개")
pd.DataFrame(답["recommendations"])                        # 추천 목록 → 표

상태 코드 : 200
답에 든 것 : ['user_id', 'persona', 'recommendations']
투자자 U0001 · 투자성향 안정형 · 추천 10개


,rank,item_id,item_name,risk_level,score
0,1,I002,TIGER 미국배당다우존스,2,0.6211
1,2,I043,Vanguard FTSE Developed Markets ETF,3,0.4705
2,3,I014,KT,2,0.3974
3,4,I011,KODEX 단기채권,1,0.3722
4,5,I021,Verizon Communications,2,0.3690
5,6,I015,LG유플러스,2,0.3645
6,7,I033,TIGER 은행고배당플러스TOP10,2,0.3356
7,8,I020,Johnson & Johnson,2,0.3127
8,9,I035,TIGER 200,3,0.3036
9,10,I055,Apple,3,0.2801


**결과 읽는 법**

- **상태 코드 200** — 성공입니다.
- **답에 든 것** — `user_id` · `persona` · `recommendations` 세 가지입니다.
- 아래 표는 `recommendations` 를 표로 바꾼 것입니다. U0001(안정형)의 추천 **10개**이고, `risk_level` 은 모두 **3 이하**입니다.
  - 1등은 TIGER 미국배당다우존스(위험도 2, 점수 0.6211) — 7주차 7.4 에서 본 U0001 의 1등과 같습니다.

In [6]:
# 이 셀에서 하는 일 — API 의 답이 7주차 결과 파일과 같은지 비교한다
# 왜 하나 — API 가 추천을 바꾸지 않고 파일 그대로 꺼내 주는지 확인한다

파일 = pd.read_csv("output/recommendations.csv", encoding="utf-8-sig")
파일_U0001 = list(파일[파일["user_id"] == "U0001"]["item_id"])            # 파일에서 읽은 U0001 의 10개

API_U0001 = []
for 추천 in 답["recommendations"]:                                       # API 가 준 10개
    API_U0001.append(추천["item_id"])

print("파일 :", 파일_U0001)
print("API  :", API_U0001)
print("같은가 :", 파일_U0001 == API_U0001)

파일 : ['I002', 'I043', 'I014', 'I011', 'I021', 'I015', 'I033', 'I020', 'I035', 'I055']
API  : ['I002', 'I043', 'I014', 'I011', 'I021', 'I015', 'I033', 'I020', 'I035', 'I055']
같은가 : True


**결과 읽는 법**

- 두 줄의 종목 번호 10개가 **순서까지 같고**, 마지막 줄이 **True** 입니다.
- API 는 추천을 새로 계산하지 않고 7주차 파일을 **그대로 꺼내 줍니다.** 이것이 서빙입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.4 Swagger — 코드 없이 버튼으로 요청하기 (강사 시연)</mark>

**8.4 에서 하는 일** — FastAPI 가 자동으로 만든 **Swagger 화면**(`/docs`)을 새 탭으로 열고, 버튼으로 U0001 의 추천을 요청해 봅니다.

> 🎥 **수업에서는 강사가 화면으로 보여 줍니다.** 복습할 때 직접 따라 해 보세요.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- API 를 쓰는 사람(다른 팀 개발자)은 「어떤 주소에 무엇을 보내면 무엇이 오는지」를 알아야 합니다. Swagger 가 그 **설명서**입니다.
- 코드를 쓰지 않아도 버튼으로 **직접 시험**해 볼 수 있습니다.

**왜 중요한가**
- 설명서를 사람이 따로 쓰면 코드가 바뀔 때 설명서가 낡습니다. Swagger 는 **코드에서 자동으로** 만들어지므로 늘 코드와 같습니다.
- 실무에서는 API 를 만들면 이 화면 주소부터 다른 팀에 알려 줍니다.

**알아 둘 배경 지식**
- **OpenAPI** — API 설명서를 쓰는 국제 표준 형식. Swagger 는 그 설명서를 보여 주는 화면입니다
- 오늘 API 코드의 함수 설명(`"""..."""`)과 `description=` 이 Swagger 화면에 그대로 나옵니다

**어떻게 따라 하나**
1. 아래 셀을 실행하면 **새 탭**이 열립니다 (Colab). 안 열리면 셀 아래에 생긴 링크를 누릅니다
2. `GET /recommend/{user_id}` 줄을 누릅니다
3. 오른쪽 **`Try it out`** 버튼을 누릅니다
4. `user_id` 칸에 `U0001` 을 적고, **`Execute`** 를 누릅니다
5. 아래 **Response body** 에 8.3 과 같은 JSON 이, **Code** 에 200 이 나옵니다

In [7]:
# 이 셀에서 하는 일 — Swagger 화면(/docs)을 새 탭으로 연다
# 왜 하나 — 코드 없이 버튼으로 API 에 요청을 보내 본다

if "google.colab" in sys.modules:                          # Colab 이면
    from google.colab import output                        # Colab 의 화면 도구
    output.serve_kernel_port_as_window(포트, path="/docs")  # 8765 번 서버의 /docs 를 새 탭 링크로
else:                                                      # 내 컴퓨터면
    print(f"브라우저 주소창에 {주소}/docs 를 입력하세요")

브라우저 주소창에 http://localhost:8765/docs 를 입력하세요


**결과 읽는 법**

- Colab 에서는 셀 아래에 **링크**가 생깁니다. 누르면 새 탭에 「추천 API」 Swagger 화면이 열립니다.
  - 이 링크는 **노트북을 실행한 본인만** 열 수 있습니다. 다른 사람에게 보내도 열리지 않습니다.
- 화면 맨 위에 API 이름과 설명, 그 아래 `GET /health` · `GET /recommend/{user_id}` 두 줄이 보입니다. 8.1 의 엔드포인트 두 개입니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">🎯 8.5 없는 번호 — 404 로 「없다」고 알려 준다</mark>

**8.5 에서 하는 일** — 결과 파일에 없는 번호 **U9999** 로 요청을 보내 봅니다.

---
> 📖 **여기부터 「자세히」 전까지는 복습할 때 읽는 부분입니다.** 수업에서는 건너뜁니다.

**왜 배우나**
- API 는 늘 성공만 하지 않습니다. **잘못된 요청에 어떻게 답할지**도 API 의 약속입니다.

**왜 중요한가**
- 답이 없을 때 빈 목록 `[]` 을 돌려주면, 앱은 「추천할 게 없는 고객」인지 「번호가 틀렸는지」 구별할 수 없습니다.
- **상태 코드**로 「성공 / 없음 / 잘못된 요청」을 나눠 주면, 앱이 상황에 맞는 화면(「고객 번호를 확인해 주세요」)을 보여 줄 수 있습니다.

**알아 둘 배경 지식 — 자주 보는 상태 코드**

| 코드 | 뜻 | 오늘 언제 |
|---|---|---|
| 200 | 성공 (OK) | 8.2 · 8.3 |
| 404 | 찾는 것이 없음 (Not Found) | 지금 — U9999 |
| 422 | 요청이 규칙에 맞지 않음 (Unprocessable) | 8.6 — `k=20` |
| 500 | 서버 안에서 오류 (Internal Server Error) | API 코드에 문제가 있을 때 |

- 웹 서핑 중 「404 Not Found」 화면을 본 적이 있다면, 그것도 같은 404 입니다.

**어떻게 동작하나**
1. `/recommend/U9999` 로 요청을 보낸다
2. API 가 결과 파일에서 U9999 의 줄을 찾지만 0줄이다
3. `HTTPException(status_code=404, ...)` 으로 404 와 설명을 돌려준다

In [8]:
# 이 셀에서 하는 일 — 결과 파일에 없는 번호로 요청을 보낸다
# 왜 하나 — API 가 「없다」를 어떻게 알려 주는지 본다

응답 = requests.get(주소 + "/recommend/U9999")             # 없는 투자자 번호
print("상태 코드 :", 응답.status_code)
print("응답 :", 응답.json())

상태 코드 : 404
응답 : {'detail': 'U9999 는 추천이 없는 투자자입니다'}


**결과 읽는 법**

- **상태 코드 404** — 찾는 것이 없다는 뜻입니다.
- **응답 `{'detail': 'U9999 는 추천이 없는 투자자입니다'}`** — API 코드의 `HTTPException` 에 적어 둔 설명이 그대로 왔습니다.
- 빈 목록이 아니라 **오류 번호와 이유**를 돌려줬습니다.

**🗣 수업에서 함께 보기**

오늘 API 는 7주차 CSV 파일을 **그대로 꺼내 줄 뿐**입니다. 8.3 에서 확인했듯 내용도 파일과 똑같습니다.

그렇다면 고객 앱이 CSV 파일을 **직접 읽으면** 되지 않을까요? 왜 굳이 API 를 사이에 둘까요?

- 힌트 — 앱이 하나가 아니라 **여러 개**(모바일 앱 · 상담 화면 · 챗봇)라면? 그리고 CSV 파일에는 **300명 모두의** 추천이 들어 있다는 점을 생각해 보세요.
- 수업에서 함께 이야기합니다. 답은 노트북 맨 아래 **「답」**에도 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.6 더 해 보기 — 개수 정하기 · 규칙 어기기 · 응답 시간</mark>

> 📖 **혼자 읽기** — 복습할 때 직접 돌려 보는 부분입니다.

**8.6 에서 하는 일** — 세 가지를 더 해 봅니다.
1. `?k=3` — 추천을 **3개만** 받기
2. `?k=20` — 규칙(1 \~ 10)을 어긴 요청 → **422**
3. 요청 100번의 평균 **응답 시간** — 7주차 「서빙은 가볍다」를 API 로 확인

**알아 둘 배경 지식**
- **쿼리 파라미터 (query parameter)** — 주소 뒤 `?` 다음에 붙이는 선택 값. `?k=3` 이면 `k` 가 3
- **입력 검증 (validation)** — 요청 값이 규칙에 맞는지 검사하는 것. FastAPI 는 코드의 `Query(10, ge=1, le=10)` 을 보고 **자동으로** 검사합니다

In [9]:
# 이 셀에서 하는 일 — k=3 · k=20 요청을 보내고, 요청 100번의 평균 응답 시간을 잰다
# 왜 하나 — 쿼리 파라미터 · 입력 검증 · 서빙의 가벼움을 직접 본다

응답 = requests.get(주소 + "/recommend/U0001?k=3")
print(f"k=3  → 상태 코드 {응답.status_code} · 추천 {len(응답.json()['recommendations'])}개")

응답 = requests.get(주소 + "/recommend/U0001?k=20")
print(f"k=20 → 상태 코드 {응답.status_code} · {응답.json()['detail'][0]['msg']}")

시작 = time.perf_counter()
for _ in range(100):                                       # 같은 요청을 100번
    requests.get(주소 + "/recommend/U0001")
print(f"요청 한 번 평균 {(time.perf_counter() - 시작) / 100 * 1000:.1f}ms")

k=3  → 상태 코드 200 · 추천 3개
k=20 → 상태 코드 422 · Input should be less than or equal to 10


요청 한 번 평균 2.5ms


**결과 읽는 법**

- **k=3** → 200, 추천 **3개**. 같은 엔드포인트라도 쿼리 파라미터로 받을 개수를 바꿀 수 있습니다.
- **k=20** → **422** 와 `Input should be less than or equal to 10`(10 이하여야 한다)
  - API 코드에 검사하는 줄을 따로 쓰지 않았는데, FastAPI 가 `le=10` 을 보고 막았습니다.
- **요청 한 번 평균** — 몇 ms 정도입니다(컴퓨터마다 다릅니다). 학습 없이 파일에서 꺼내기만 하니 빠릅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.7 배포 개념 — 이 컴퓨터 안의 API 를 인터넷으로</mark>

> 📖 **혼자 읽기** — 수업에서는 PPT 로 설명합니다. 오늘은 실습하지 않습니다.

**1. 배포 (deployment) 란**
- 지금 API 는 `localhost` — **이 Colab 컴퓨터 안**에서만 부를 수 있습니다. Colab 을 닫으면 API 도 꺼집니다.
- **배포**는 API 를 **늘 켜져 있는 서버 컴퓨터**에 올려, **인터넷 주소**로 부를 수 있게 하는 일입니다.

| | 지금 (`localhost`) | 배포한 뒤 |
|---|---|---|
| 누가 부를 수 있나 | 이 Colab 안의 프로그램만 | 인터넷 주소를 아는 프로그램 |
| 언제 켜져 있나 | Colab 을 열어 둔 동안만 | 24시간 |
| 주소 | `http://localhost:8765` | 예: `https://api.회사이름.com` |

**2. 배포하면 함께 생기는 일**
- **결과 파일 갱신** — 매일 밤 7주차 스크립트가 새 파일을 만들면, API 가 새 파일을 다시 읽게 해야 합니다
- **장애 대비** — 서버가 죽으면 자동으로 다시 켜고, `/health` 로 계속 확인합니다
- **접속 기록** — 누가 언제 어떤 요청을 보냈는지 남깁니다

**3. 금융에서 특히 중요한 것 — 인증 (authentication)**
- 오늘 API 는 **번호만 알면 누구나** 그 사람의 추천과 투자성향을 볼 수 있습니다.
- 투자성향은 고객의 **개인정보**입니다. 실제 서비스에서는 「요청한 사람이 그 고객 본인인가(또는 권한 있는 직원인가)」를 먼저 확인해야 합니다.
- 이렇게 요청한 사람을 확인하는 것을 **인증**이라고 합니다. 금융 회사의 API 는 인증 없이 열지 않습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📖 8.8 오늘의 마무리</mark>

1. 7주차 결과 파일을 읽어 추천을 돌려주는 **API** 를 FastAPI 로 만들었습니다 — 엔드포인트 `/health` · `/recommend/{user_id}`.
2. API 를 **서버**로 켜 두고, `/health` 에 요청해 **200** 과 `rows: 3000` 을 받았습니다.
3. `/recommend/U0001` 에 요청해 **JSON** 으로 추천 10개를 받았고, 7주차 파일과 **순서까지 같았습니다.**
4. **Swagger**(`/docs`)에서 코드 없이 버튼으로 같은 요청을 보낼 수 있었습니다.
5. 없는 번호에는 **404**, 규칙을 어긴 요청에는 **422** 로 답했습니다.

**오늘 기억할 한 줄** — **API 는 추천을 계산하지 않고, 만들어 둔 추천을 약속된 모양으로 꺼내 줍니다.**

다음 주에는 이 추천을 **사람이 보는 화면**(Streamlit 대시보드)으로 만들고, 투자자별로 비교 · 해석해 봅니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">📝 혼자 해 보기</mark>

수업이 끝난 뒤 혼자 풀어 보는 문제입니다. 문제마다 **어느 절의 결과를 보면 되는지** 제목에 적어 두었습니다.

답은 바로 아래 **「답」**에 있습니다. 먼저 스스로 생각해 본 뒤에 읽어 보세요.

**📝 혼자 해 보기 1 · 8.5 결과**

U9999 처럼 없는 번호가 오면 API 는 **404 오류**를 돌려줍니다. 그냥 **빈 목록**(`"recommendations": []`)과 200 을 돌려주면 안 될까요?

- 힌트 — 고객 앱 입장에서 「이 고객은 추천할 종목이 없다」와 「고객 번호가 잘못됐다」를 구별해야 할까요?
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 2 · 8.2 결과**

API 는 **켜질 때 결과 파일을 한 번만** 읽습니다. 오늘 밤 7주차 스크립트가 새 결과 파일을 만들었다면, 이미 켜져 있는 API 는 새 추천을 돌려줄까요?

- 힌트 — 8.1 의 「어떻게 동작하나」 1번과, `recommend_api.py` 맨 위 설명의 마지막 줄을 읽어 보세요.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

**📝 혼자 해 보기 3 · 8.7 내용**

오늘 API 를 인증 없이 그대로 인터넷에 배포했다고 해 봅시다. 누군가 `/recommend/U0001` 부터 `/recommend/U0300` 까지 차례로 요청을 보낸다면 어떤 문제가 생길까요?

- 힌트 — 8.3 의 답에 `persona`(투자성향)가 들어 있었습니다.
- 답은 노트북 맨 아래 **「답」**에 있습니다.

### <mark style="display:block; background:#f8bbd0; color:#1a1a1a; padding:6px 12px; border-radius:4px">답</mark>

먼저 스스로 답을 생각해 본 뒤에 읽어 보세요. 답에 나오는 숫자는 모두 이 노트북을 실제로 실행해서 얻은 값입니다.

**함께 보기 답 — 여러 프로그램이 같은 추천을 받게 하고, 필요한 한 사람 것만 내주고, 파일이 바뀌어도 앱을 고치지 않게 하려는 것입니다**

- **여러 프로그램이 같은 답을 받습니다**
  - 모바일 앱 · 상담 화면 · 챗봇이 각자 파일을 읽으면, 누군가는 어제 파일을, 누군가는 오늘 파일을 읽을 수 있습니다.
  - API 하나를 거치면 모두 **같은 시점의 같은 추천**을 봅니다.
- **필요한 것만 내줍니다**
  - CSV 에는 **300명 모두**의 추천과 투자성향이 들어 있습니다. 앱에 파일을 통째로 주면 다른 고객의 정보까지 넘어갑니다.
  - API 는 요청한 **한 사람의 10줄만** 돌려줍니다.
- **앱이 파일 사정을 몰라도 됩니다**
  - 파일 이름 · 위치 · 열 순서가 바뀌거나, 나중에 CSV 대신 데이터베이스를 쓰게 돼도 **주소와 답의 모양만 그대로면** 앱은 고칠 필요가 없습니다.
- **잘못된 요청을 한곳에서 막습니다** — 없는 번호는 404, 규칙 위반은 422 (8.5 · 8.6)
- **한 줄로 정리하면** — API 는 추천과 그것을 쓰는 프로그램 사이의 **약속된 창구**입니다.

**혼자 해 보기 1 답 — 구별해야 합니다. 둘은 고객에게 보여 줄 화면이 다르기 때문입니다**

- **「추천할 종목이 없다」** — 정상적인 상황입니다. 앱은 「오늘은 추천이 없습니다」나 인기 종목을 보여 주면 됩니다.
- **「고객 번호가 잘못됐다」** — 어딘가 오류가 난 상황입니다. 앱은 「고객 정보를 확인해 주세요」를 보여 주고, 개발팀은 왜 틀린 번호가 왔는지 찾아야 합니다.
- 빈 목록과 200 을 돌려주면 두 상황이 **똑같아 보여서** 오류를 놓칩니다.
- 404 로 나눠 주면 앱이 상황에 맞게 반응하고, 접속 기록에서 404 가 많아지면 문제를 빨리 알아챌 수 있습니다.
- **한 줄로 정리하면** — 상태 코드는 「무엇이 잘못됐는지」를 알려 주는 약속입니다. 실패도 정확히 알려 줘야 합니다.

**혼자 해 보기 2 답 — 돌려주지 않습니다. 켜질 때 읽은 옛 파일을 계속 씁니다. API 를 다시 켜야 새 추천이 나갑니다**

- `recommend_api.py` 의 `추천표 = pd.read_csv(...)` 는 함수 밖에 있어서 **API 가 켜질 때 한 번만** 실행됩니다.
  - 그 뒤로는 메모리에 올려 둔 `추천표` 에서 꺼내 줄 뿐, 파일을 다시 열지 않습니다.
- 그래서 파일이 새로 만들어져도, 켜져 있는 API 는 **어제 추천**을 계속 돌려줍니다.
- 실무에서는 보통 둘 중 하나를 합니다.
  - 7주차 스크립트가 끝나면 **API 를 다시 켠다**(재시작)
  - API 가 정해진 시각마다 파일이 바뀌었는지 보고 **다시 읽는다**
- 이것도 파이프라인의 한 단계입니다 — 「결과 파일 저장」 다음에 「API 가 새 파일을 읽기」가 이어져야 합니다.
- **한 줄로 정리하면** — 파일을 새로 만드는 것과, API 가 그 파일을 쓰게 하는 것은 별개의 단계입니다.

**혼자 해 보기 3 답 — 300명 전원의 투자성향과 추천이 그대로 새어 나갑니다**

- 번호는 `U0001` 부터 `U0300` 까지 **차례대로** 붙어 있어서, 하나씩 바꿔 가며 요청하면 300명 모두를 알아낼 수 있습니다.
- 응답에는 **투자성향**(안정형 · 성장형 · 공격형)이 들어 있습니다. 고객이 설문으로 밝힌 금융 **개인정보**입니다.
- 추천 목록으로 그 고객이 어떤 종목을 담지 **않았는지**도 짐작할 수 있습니다(이미 담은 종목은 추천에서 빠지므로).
- 그래서 실제 금융 API 는
  - **인증** — 요청한 사람이 그 고객 본인이거나 권한 있는 직원인지 확인한다
  - **번호 대신 짐작할 수 없는 값**을 쓰거나, 요청 횟수를 제한한다
  - **접속 기록**을 남겨 이상한 요청을 찾아낸다
- **한 줄로 정리하면** — API 를 인터넷에 여는 순간, 「누가 부를 수 있는가」가 가장 먼저 정해져야 합니다.